In [21]:
import pandas as pd
import numpy as np

customers = pd.read_csv('../data/cleaned/olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
orders = pd.read_csv('../data/cleaned/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/cleaned/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/cleaned/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/cleaned/olist_order_reviews_dataset.csv')
order_level_reviews = pd.read_csv('../data/cleaned/order_level_reviews.csv')
products = pd.read_csv('../data/cleaned/olist_products_dataset.csv')
category_translation = pd.read_csv('../data/cleaned/product_category_name_translation.csv')
sellers = pd.read_csv('../data/cleaned/olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': str})
geo_lookup = pd.read_csv('../data/cleaned/geo_lookup.csv', dtype={'geolocation_zip_code_prefix': str})

# Rebuild purchaseMonth and orderOverTime — needed downstream, not
# inherited from 04's cells.
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['purchaseMonth'] = orders['order_purchase_timestamp'].dt.to_period('M')
orderOverTime = orders.groupby('purchaseMonth')['order_id'].count()


In [24]:
a=customers.merge(
    orders,
    on='customer_id',
    how='left'
)
a['firstPurchase']=a.groupby('customer_unique_id')['purchaseMonth'].transform('min')
b=a[a['firstPurchase']==a['purchaseMonth']]

repeatCustomer=a.groupby('customer_unique_id')['order_id'].count()
repeatCustomer=repeatCustomer[repeatCustomer>=2]
# How many customers have more than one row in b (multiple orders in their first month)?
print(b['customer_unique_id'].duplicated().sum())

# Keep one row per customer - their first order's lateDelivery status
bFirstOrder = b.drop_duplicates('customer_unique_id', keep='first')[['customer_unique_id', 'lateDelivery']]

# Flag whether each customer went on to become a repeat customer
bFirstOrder['isRepeat'] = bFirstOrder['customer_unique_id'].isin(repeatCustomer.index)

# Compare repeat rate by first-order delivery status
repeatRateByFirstStatus = bFirstOrder.groupby('lateDelivery')['isRepeat'].mean() * 100
print(repeatRateByFirstStatus)

1322
lateDelivery
Late Delivery       2.619455
On Time Delivery    3.126022
Unknown             4.226336
Name: isRepeat, dtype: float64


In [25]:
print(bFirstOrder['lateDelivery'].value_counts())
print(b[b['lateDelivery'] == 'Unknown']['order_status'].value_counts())

lateDelivery
On Time Delivery    85636
Late Delivery        7597
Unknown              2863
Name: count, dtype: int64
order_status
shipped        1097
unavailable     598
canceled        577
invoiced        312
processing      298
delivered        29
created           5
approved          2
Name: count, dtype: int64


In [26]:
b['customer_unique_id'].duplicated().sum()

np.int64(1322)

In [27]:
a=customers.merge(
    orders,
    on='customer_id',
    how='left'
)
b=a.merge(
    order_items,
    on='order_id',
    how='left'
)
sparseSeller=['TO','AL','RR','AP']
stateWiseRev=b.groupby('customer_state')['item_total_value'].sum()
sellerInSparse=stateWiseRev.index.isin(sparseSeller)
sparseSellerRev=stateWiseRev[sellerInSparse].sum()

In [28]:
c=(sparseSellerRev/stateWiseRev.sum())*100
c

np.float64(1.1607954176319604)

In [29]:
stateWiseRev

customer_state
AC      19669.70
AL      96229.40
AM      27835.73
AP      16262.80
BA     611506.67
CE     275606.30
DF     353229.44
ES     324801.91
GO     347706.93
MA     151171.99
MG    1856161.49
MS     135956.67
MT     186168.96
PA     217647.11
PB     140987.81
PE     322237.69
PI     108132.28
PR     800935.44
RJ    2129681.98
RN     101895.08
RO      57558.02
RR      10064.62
RS     885826.76
SC     610213.60
SE      73032.32
SP    5921678.12
TO      61354.42
Name: item_total_value, dtype: float64

In [30]:
sellerInSparse

array([False,  True, False,  True, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False,  True, False, False, False, False,  True])

In [31]:
sparseSellerRev

np.float64(183911.24)

In [32]:
b.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state', 'order_id', 'order_status',
       'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'lateDelivery', 'validDateCheck',
       'delivery_days', 'delivery_delay_days', 'purchaseMonth',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'priceFreightCheck', 'item_total_value'],
      dtype='str')

In [33]:
b['priceFreightCheck'].value_counts()

priceFreightCheck
Normal                108143
Delivery Expensive      4124
Free Delivery            383
Name: count, dtype: int64

In [34]:
b.groupby('priceFreightCheck')['priceFreightCheck'].count()

priceFreightCheck
Delivery Expensive      4124
Free Delivery            383
Normal                108143
Name: priceFreightCheck, dtype: int64

In [35]:
freightExpensiveLateRate = b[b['priceFreightCheck'] == 'Delivery Expensive'].groupby('lateDelivery')['lateDelivery'].count()
freightExpensiveLateRate = freightExpensiveLateRate / freightExpensiveLateRate.sum() * 100
print(freightExpensiveLateRate)

lateDelivery
Late Delivery        8.923375
On Time Delivery    88.118332
Unknown              2.958293
Name: lateDelivery, dtype: float64


In [36]:
sellersItemSold = order_items.merge(sellers, on='seller_id', how='left')
sellersItemPdtSold = sellersItemSold.merge(products, on='product_id', how='left')
sellerItemPdtSoldDelivery = sellersItemPdtSold.merge(orders, on='order_id', how='left')

sellerReviews = sellerItemPdtSoldDelivery.merge(
    order_level_reviews[['order_id', 'review_score_avg']],
    on='order_id',
    how='left'
)
firstSaleMonth = sellerItemPdtSoldDelivery.groupby('seller_id')['purchaseMonth'].min()

sellerTenure = pd.DataFrame({'firstSaleMonth': firstSaleMonth})
sellerTenure['firstSaleOrdinal'] = sellerTenure['firstSaleMonth'].astype('int64')
sellerTenure['tenureTier'] = pd.qcut(sellerTenure['firstSaleOrdinal'], q=3, labels=['Oldest', 'Mid', 'Newest'])

sellerTenurePerf = sellerReviews.groupby('seller_id').agg(
    totalRev=('item_total_value', 'sum'),
    avgReviewScore=('review_score_avg', 'mean'),
    avgDelay=('delivery_delay_days', 'mean')
)
sellerTenurePerf = sellerTenurePerf.merge(sellerTenure[['tenureTier']], on='seller_id', how='left')

# D.4. Consolidated seller scorecard - revenue, review, delay, tenure
top50Sellers = sellerTenurePerf.sort_values('totalRev', ascending=False).head(50).index

sellerTenurePerf['highRevenue'] = sellerTenurePerf.index.isin(top50Sellers)
sellerTenurePerf['goodReview'] = sellerTenurePerf['avgReviewScore'] > 4
sellerTenurePerf['goodDelay'] = sellerTenurePerf['avgDelay'] <= sellerTenurePerf['avgDelay'].median()   # adjust cutoff if you want it looser

print(sellerTenurePerf[['highRevenue', 'goodReview', 'goodDelay']].sum())

strongAll = sellerTenurePerf[sellerTenurePerf['highRevenue'] & sellerTenurePerf['goodReview'] & sellerTenurePerf['goodDelay']]
print('Strong on all 3:', strongAll.shape[0])

highRevWeakOther = sellerTenurePerf[sellerTenurePerf['highRevenue'] & ~(sellerTenurePerf['goodReview'] & sellerTenurePerf['goodDelay'])]
print('High revenue but weak on review/delay:', highRevWeakOther.shape[0])
print(highRevWeakOther[['totalRev', 'avgReviewScore', 'avgDelay']])

highRevenue      50
goodReview     1775
goodDelay      1493
dtype: int64
Strong on all 3: 16
High revenue but weak on review/delay: 34
                                   totalRev  avgReviewScore   avgDelay
seller_id                                                             
04308b1ee57b6625f47df1d56f00eedf   63184.99        3.882979 -11.576087
1025f0e2d44d7041d6cf58b6550e0bfa  172860.69        3.853107 -11.104930
17e34d8224d27a541263c4c64b11a56b   61603.20        4.139098 -10.853846
1f50f920176fa81dab994f9023523100  142104.98        3.987259 -10.852544
218d46b86c1881d022bce9c68a7d4b15   62737.54        4.149412 -11.699284
25c5c91f63607446a97b143d2d535d31   61679.33        3.718284 -11.966165
391fc6631aebcf3004804e51b40bcf1e   50142.83        3.961349 -11.535774
4869f7a5dfa277a7dca6462dcf3b52b2  249640.70        4.122822 -11.164634
4a3ca9315b744ce9f8e9374361493884  235539.96        3.803262  -9.644433
4e922959ae960d389249c378d1c939f5   53304.75        3.895323 -13.029148
522620dcb18a6

In [37]:
highRevWeakOther['failedReview'] = ~highRevWeakOther['goodReview']
highRevWeakOther['failedDelay'] = ~highRevWeakOther['goodDelay']
print(highRevWeakOther[['avgReviewScore', 'avgDelay', 'failedReview', 'failedDelay']])

                                  avgReviewScore   avgDelay  failedReview  \
seller_id                                                                   
04308b1ee57b6625f47df1d56f00eedf        3.882979 -11.576087          True   
1025f0e2d44d7041d6cf58b6550e0bfa        3.853107 -11.104930          True   
17e34d8224d27a541263c4c64b11a56b        4.139098 -10.853846         False   
1f50f920176fa81dab994f9023523100        3.987259 -10.852544          True   
218d46b86c1881d022bce9c68a7d4b15        4.149412 -11.699284         False   
25c5c91f63607446a97b143d2d535d31        3.718284 -11.966165          True   
391fc6631aebcf3004804e51b40bcf1e        3.961349 -11.535774          True   
4869f7a5dfa277a7dca6462dcf3b52b2        4.122822 -11.164634         False   
4a3ca9315b744ce9f8e9374361493884        3.803262  -9.644433          True   
4e922959ae960d389249c378d1c939f5        3.895323 -13.029148          True   
522620dcb18a6b31cd7bdf73665113a9        3.897727 -14.016854          True   

In [39]:
repeatRateByFirstStatus.to_csv('../data/summary/repeat_rate_by_first_status.csv', header=True)
sellerTenurePerf.to_csv('../data/summary/seller_scorecard.csv')